# Camera Discovery Harvest URLs Test Notebook

This Google Colab notebook tests the `camera-discovery harvest-urls` CLI workflow. Harvest mode is extraction-only: it bypasses target resolution, geocoding, validation, trust policy, scope enforcement, LLM review, GeoJSON, map output, `cameras.md`, and review ZIP generation.

When source metadata already contains camera coordinates, orientation/direction, date, time, or timestamps, harvest mode saves those values alongside the URL in `camera_urls.csv` and `camera_urls.jsonl`. The notebook displays those promoted fields but does not infer or geocode missing values.

The notebook is a CLI harness only. It does not implement harvest source logic and does not patch application source files at runtime.


In [ ]:
# Colab setup: run from the checked-out repository root.
from pathlib import Path
import os

REPO_ROOT = Path.cwd()
print('Repository root:', REPO_ROOT)
print('pyproject.toml exists:', (REPO_ROOT / 'pyproject.toml').exists())


In [ ]:
# Install the package in editable mode.
%pip install -e .


In [ ]:
# Verify CLI registration.
!camera-discovery --help
!camera-discovery harvest-urls --help


## Harvest all supported media types

This command collects raw direct media/camera URLs across supported media categories and writes plain URL and metadata outputs. It does not validate streams or write inventory artifacts.


In [ ]:
!camera-discovery harvest-urls "California traffic cameras" \
  --output-dir runs/harvest-california \
  --max-urls 10000 \
  --discovery-mode both \
  --enable-browser-capture


## Harvest only HLS / `.m3u8` URLs


In [ ]:
!camera-discovery harvest-urls "California traffic cameras" \
  --output-dir runs/harvest-california-hls \
  --max-urls 10000 \
  --media .m3u8


## Harvest a media mix: HLS, images, and generic stream URLs


In [ ]:
!camera-discovery harvest-urls "California traffic cameras" \
  --output-dir runs/harvest-california-media-mix \
  --max-urls 10000 \
  --media hls,image,stream


## Inspect output paths, summary counts, and promoted metadata fields

The summary includes counts for records with collected coordinates, orientation, and date/time metadata. The CSV/JSONL outputs include top-level fields such as `lat`, `lon`, `coordinate_source`, `direction`, `bearing`, `heading`, `date`, `time`, and `timestamp` when those values were present in source metadata.


In [ ]:
from pathlib import Path
import csv
import json

HARVEST_DIR = Path('runs/harvest-california')
paths = {
    'camera_urls.txt': HARVEST_DIR / 'camera_urls.txt',
    'camera_urls.csv': HARVEST_DIR / 'camera_urls.csv',
    'camera_urls.jsonl': HARVEST_DIR / 'camera_urls.jsonl',
    'harvest_summary.json': HARVEST_DIR / 'harvest_summary.json',
    'source_rows.jsonl': HARVEST_DIR / 'source_rows.jsonl',
}
for label, path in paths.items():
    print(f'{label}: {path} exists={path.exists()} size={path.stat().st_size if path.exists() else 0}')

summary_path = paths['harvest_summary.json']
if summary_path.exists():
    summary = json.loads(summary_path.read_text())
    print('\nSummary counts:')
    for key in [
        'media_filter',
        'raw_records',
        'unique_urls',
        'pre_filter_unique_urls',
        'media_filtered_urls',
        'written_urls',
        'records_with_coordinates',
        'records_with_orientation',
        'records_with_datetime',
        'max_urls',
        'unlimited',
    ]:
        print(f'{key}:', summary.get(key))
    print('by_media_type:', summary.get('by_media_type', {}))
    print('by_source_host:', summary.get('by_source_host', {}))

csv_path = paths['camera_urls.csv']
if csv_path.exists():
    with csv_path.open(newline='', encoding='utf-8') as f:
        rows = list(csv.DictReader(f))
    promoted_columns = ['url', 'media_type', 'lat', 'lon', 'coordinate_source', 'direction', 'bearing', 'heading', 'date', 'time', 'timestamp']
    rows_with_promoted_metadata = [
        row for row in rows
        if row.get('lat') or row.get('lon') or row.get('direction') or row.get('bearing') or row.get('heading') or row.get('date') or row.get('time') or row.get('timestamp')
    ]
    print(f'\nRows with promoted coordinate/orientation/date-time metadata: {len(rows_with_promoted_metadata)}')
    for row in rows_with_promoted_metadata[:10]:
        print({key: row.get(key) for key in promoted_columns})

jsonl_path = paths['camera_urls.jsonl']
if jsonl_path.exists():
    print('\nFirst JSONL records with promoted metadata:')
    shown = 0
    for line in jsonl_path.read_text(encoding='utf-8').splitlines():
        if not line.strip():
            continue
        record = json.loads(line)
        if any(record.get(key) is not None for key in ['lat', 'lon', 'direction', 'bearing', 'heading', 'date', 'time', 'timestamp']):
            print(json.dumps({key: record.get(key) for key in ['url', 'media_type', 'lat', 'lon', 'coordinate_source', 'direction', 'bearing', 'heading', 'date', 'time', 'timestamp']}, indent=2))
            shown += 1
            if shown >= 5:
                break
    if shown == 0:
        print('No promoted metadata fields found in the first harvest output. This means the discovered source metadata did not include those fields, not that harvest inferred missing values.')


## Print the first harvested URLs


In [ ]:
N = 25
url_path = Path('runs/harvest-california/camera_urls.txt')
if url_path.exists():
    for idx, line in enumerate(url_path.read_text().splitlines()[:N], start=1):
        print(f'{idx:03d}: {line}')
else:
    print('No camera_urls.txt file found yet.')


## Optional: zip and download harvest outputs


In [ ]:
from pathlib import Path
import shutil

out_dir = Path('runs/harvest-california')
if out_dir.exists():
    archive = shutil.make_archive(str(out_dir), 'zip', root_dir=out_dir)
    print('Created:', archive)
    try:
        from google.colab import files
        files.download(archive)
    except Exception as exc:
        print('Download helper unavailable outside Colab:', exc)
else:
    print('Run harvest first; output directory does not exist:', out_dir)
